# Magnetic Resonance Imaging II

## Exercise 1

---

**Prof. Dr. Markus Zimmermann**

**Mohamed Wassim Bensalem**

---

This notebook guides you through the full pipeline of a 2D Gradient-Echo (GRE) sequence: from sequence design and timing calculation, through simulation on a digital phantom, to image reconstruction. You will use two main libraries throughout:
 
- **PyPulseq** (`pypulseq`) — a Python toolbox for designing MR pulse sequences. It handles the low-level details of gradient waveforms, RF pulses, ADC windows, and timing constraints, and exports sequences in the vendor-neutral `.seq` file format.
- **MRzero** (`MRzeroCore`) — a differentiable MRI simulator. It takes a `.seq` file and a digital phantom as input and simulates the acquired MR signal, allowing you to test sequences without access to a physical scanner.

## Software Imports
 
All required libraries and modules are imported here.

In [ ]:
%matplotlib widget

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import pypulseq as pp
import torch

## System and Sequence Parameters

The parameters defined here control the spatial resolution and field of view of the acquisition, as well as the contrast-determining parameters TR, TE, and flip angle. `n_x` and `n_y` define both the number of k-space samples and the number of reconstructed voxels along each direction. Three contrast modes are pre-configured — T1-weighted, T2-weighted, and PD-weighted — selected by the index in the `case` line (0, 1, or 2 respectively). You will run the notebook once per mode at the end to verify that the sequence produces the expected contrast behavior.

In [ ]:
seq_filename = "gre.seq"

fov_x, fov_y = 256e-3, 256e-3
n_x, n_y = 128, 128

case = ["T1-weighted", "T2-weighted", "PD-weighted"][0]

if case == "T1-weighted":
    flip_angle_deg = 90
    tr = 250e-3
    te = 5e-3
elif case == "T2-weighted":
    flip_angle_deg = 40
    tr = 5000e-3
    te = 50e-3
elif case == "PD-weighted":
    flip_angle_deg = 40
    tr = 5000e-3
    te = 5e-3

torch.set_default_device("cpu")

## System and Sequence Object

Before designing any gradients or pulses, two foundational objects must be created.
 
The first is a `pp.Opts` object, which defines the scanner hardware limits. It is passed to all subsequent gradient and RF pulse constructors to ensure that designed waveforms respect hardware constraints. The two key limits are:
 
- **Maximum gradient amplitude** (`max_grad`): the peak field strength a gradient coil can produce, in mT/m.
- **Maximum slew rate** (`max_slew`): how quickly the gradient amplitude can ramp up or down, in T/m/s. This is the primary constraint on gradient rise times, and is clinically relevant due to peripheral nerve stimulation limits.
The values used here are representative of a mid-field clinical scanner.
 
The second is a `pp.Sequence` object, which acts as the container into which all sequence blocks (RF pulses, gradients, delays, ADC windows) are added in order. It is initialized with the `system` object so that it is aware of the same hardware constraints — this is required for internal timing checks and for writing a valid `.seq` file.

In [ ]:
system = pp.Opts(
    max_grad=28,
    grad_unit="mT/m",
    max_slew=150,
    slew_unit="T/m/s",
)

seq = pp.Sequence(system)

## Task 1: RF-Pulse

The GRE sequence begins with a radiofrequency excitation pulse that tips the longitudinal magnetization by the flip angle α into the transverse plane. In PyPulseq, RF pulses are created by constructor functions that return a pulse object encapsulating the waveform shape, duration, and timing metadata used in later calculations.
 
**Your task:** Create a sinc excitation pulse using `pp.make_sinc_pulse()`. Provide the following arguments:
 
- `flip_angle` — the desired flip angle in radians. Convert from `flip_angle_deg` using `np.deg2rad()`.
- `system` — the hardware limits object. Always pass this to ensure the pulse respects scanner constraints.
- `use` — a string declaring the role of this pulse to PyPulseq. Use `"excitation"`.

Store the result in a variable called `rf`.

In [ ]:
# Your solution here

rf = pp.make_sinc_pulse(
    flip_angle=np.deg2rad(flip_angle_deg),
    system=system,
    use="excitation",
)

## Task 2: Gradient Design

This task covers the design of all gradient waveforms in the GRE sequence. In PyPulseq, gradients are trapezoidal waveforms created using `pp.make_trapezoid()`. The key parameters are:
 
- `channel` — the physical gradient axis: `"x"` (frequency encoding), `"y"` (phase encoding), or `"z"` (slice selection).
- `area` — the total gradient area in 1/m, which directly determines the k-space displacement. In PyPulseq, gradient areas are specified in units of 1/m, corresponding directly to k-space coordinates (i.e. Δk = area).
- `flat_area` / `flat_time` — alternatively, specify the area under the flat top and its duration separately. PyPulseq then computes the required gradient amplitude and rise times automatically.
- `system` — always pass the hardware limits object.

**k-space sampling:** Recall that the k-space step size is set by the FOV: Δkx = 1/FOVx and Δky = 1/FOVy. The full readout line spans `n_x` steps of Δkx, giving a total readout gradient area of `n_x · Δkx`. The pre-phaser shifts the k-space position to the start of the readout line (i.e. −kx_max), so its area is −½ of the readout gradient area.
 
**Your tasks:**
 
1. Compute the k-space step sizes `delta_kx` and `delta_ky` from the FOV parameters.
2. Create the **readout gradient** `gx` using `pp.make_trapezoid()` on channel `"x"`. Set `flat_area=n_x * delta_kx` and `flat_time=3.2e-3`.
3. Create the **readout pre-phaser** `gx_pre` on channel `"x"` with `area=-gx.area / 2`. The pre-phaser must shift k-space to exactly −kx_max, which corresponds to half the *total* k-space traversal of the readout gradient — including the area under the ramps. PyPulseq exposes this as `gx.area` (the full trapezoid area including rise and fall), which is subtly different from `gx.flat_area`. Using `gx.flat_area / 2` would leave a small error at the ramp edges.
4. Create the **ADC** (analog-to-digital converter) readout window using `pp.make_adc()`. It should acquire `n_x` samples spanning the flat top of `gx`. Set:
   - `num_samples=n_x`
   - `duration=gx.flat_time`
   - `delay=gx.rise_time` — the ADC opens once the gradient has ramped up to its flat top.
5. Create the **spoiler gradient** `gx_spoiler` on channel `"x"` with `area=4 * gx.area`. The spoiler dephases residual transverse magnetization after each readout, preventing it from contaminating the signal in subsequent TRs. Its area should be several times larger than the readout gradient area.

In [ ]:
# Your solution here

delta_kx = 1 / fov_x
delta_ky = 1 / fov_y

# flat_time = 3.2e-3
flat_time = 1.6e-3

gx = pp.make_trapezoid(
    channel="x",
    flat_area=n_x * delta_kx,
    flat_time=flat_time,
    system=system,
)

gx_rev = pp.make_trapezoid(
    channel="x",
    flat_area=-n_x * delta_kx,
    flat_time=flat_time,
    system=system,
)

gx_pre = pp.make_trapezoid(
    channel="x",
    area=-gx.area / 2,
    system=system,
)

gy_pre = pp.make_trapezoid(
    channel="y",
    area=-n_y * delta_ky / 2,
    system=system,
)

gy_blip = pp.make_trapezoid(
    channel="y",
    area=delta_ky,
    system=system,
)

adc = pp.make_adc(
    num_samples=n_x,
    duration=gx.flat_time,
    delay=gx.rise_time,
    system=system,
)

gx_spoiler = pp.make_trapezoid(
    channel="x",
    area=4 * gx.area,
    system=system,
)

## Task 3: Timing

Precise timing is critical in MRI: the echo time TE is defined as the time between the center of the RF pulse and the center of the ADC window, and the repetition time TR is the total duration of one complete sequence block (RF pulse through end of TR delay).
 
In PyPulseq, the duration of any block object (gradient, RF pulse, delay) is retrieved using `pp.calc_duration(block)`. For RF pulses specifically, the time from the *start* of the pulse to its *peak* (the RF center, where the net flip occurs) is given by `pp.calc_rf_center(rf)[0]`.
 
**TE delay:** A delay `te_delay` is inserted between the pre-phaser and the readout gradient so that the center of the ADC window falls exactly at time TE after the RF center. Work through the sequence timeline from the RF center forward to the midpoint of the readout gradient flat top, and express `te_delay` in terms of `te`, `pp.calc_duration()`, `pp.calc_rf_center()`, and the known gradient durations.
 
**TR delay:** A delay `tr_delay` fills the remainder of each TR after the spoiler gradient. Express `tr_delay` in terms of `tr` and the durations of all blocks in one TR: the RF pulse, the pre-phaser, `te_delay`, the readout gradient, and the spoiler. Note that `te_delay` must be included explicitly here — it is a real time interval in the sequence, already accounted for in the TE budget, but it still consumes time within the TR.
 
Both delays must be non-negative — a negative value means the chosen TE or TR is too short to fit the required gradient waveforms. This is enforced by the assertions below.
 
**Raster time alignment:** MR scanners operate on a discrete time grid called the gradient raster (typically 10 µs). Every block must start and end on a raster boundary, so any computed delay must be rounded up to the nearest multiple of `seq.grad_raster_time`. The small offset `raster_eps` is subtracted before rounding to avoid floating-point values that are already exactly on a boundary being spuriously rounded up to the next grid point.

In [ ]:
# Your solution here

te_delay = (
    te
    - (pp.calc_duration(rf) - pp.calc_rf_center(rf)[0])
    - pp.calc_duration(gx_pre)
    - pp.calc_duration(gx) / 2
)

tr_delay = (
    tr
    - pp.calc_duration(rf)
    - pp.calc_duration(gx_pre)
    - pp.calc_duration(gx)
    - pp.calc_duration(gx_spoiler)
    - te_delay
)

In [ ]:
raster_eps = 1e-12

te_delay = (
    np.ceil((te_delay - raster_eps) / seq.grad_raster_time) * seq.grad_raster_time
)
tr_delay = (
    np.ceil((tr_delay - raster_eps) / seq.grad_raster_time) * seq.grad_raster_time
)

assert np.all(te_delay >= 0)
assert np.all(tr_delay >= 0)

## Task 4: Sequence Assembly

With all building blocks defined, you can now assemble the full sequence. In PyPulseq, blocks are added in chronological order to the `seq` object using `seq.add_block()`. Multiple objects passed to a single `seq.add_block()` call are played out simultaneously — this is how gradients on different axes or a gradient and an ADC window are overlapped.
 
The GRE sequence repeats the following block structure once per phase-encoding step:
 
1. RF excitation pulse
2. Readout pre-phaser (`gx_pre`) and phase-encoding gradient (`gy_pre`) — played simultaneously
3. TE delay
4. Readout gradient (`gx`) and ADC — played simultaneously
5. Spoiler gradient
6. TR delay

**Your tasks:**
 
1. Compute the array of phase-encoding gradient areas `phase_areas`. Phase-encoding steps run symmetrically from −ny/2 to +ny/2 (exclusive), each separated by `delta_ky`. Use `np.linspace()` with `endpoint=False`.
2. Implement the sequence loop by iterating over `phase_areas`. For each step:
   - Add the RF pulse.
   - Create `gy_pre` using `pp.make_trapezoid()` on channel `"y"`, with `area=phase_area` and `duration=pp.calc_duration(gx_pre)`. This constrains `gy_pre` to fit within the same time slot as `gx_pre`. Add `gx_pre` and `gy_pre` simultaneously.
   - Add the TE delay using `pp.make_delay(te_delay)`.
   - Add `gx` and `adc` simultaneously.
   - Add `gx_spoiler`.
   - Add the TR delay using `pp.make_delay(tr_delay)`.

> **Note:** `gy_pre` must be recreated inside the loop for each phase-encoding step since its area changes with each iteration. All other blocks are defined once and reused.

In [ ]:
# Your solution here

seq.add_block(rf)

seq.add_block(gx_pre, gy_pre)

# seq.add_block(pp.make_delay(te_delay))

for index in range(n_y // 2):

    seq.add_block(gx, adc)

    seq.add_block(gy_blip)

    seq.add_block(gx_rev, adc)

    seq.add_block(gy_blip)

# seq.add_block(gx_spoiler)

# seq.add_block(pp.make_delay(tr_delay))

## Verification and Export
 
Once the sequence is assembled, PyPulseq provides a built-in timing checker (`seq.check_timing()`) that verifies all blocks are internally consistent and correctly aligned to the hardware raster times. Any violations are printed as an error report.
 
The sequence diagram is plotted over one TR to allow visual inspection of the gradient waveforms, RF pulse, and ADC window. Verify that the relative timing of the gradients and ADC matches what you expect from the GRE sequence diagram. Finally, the sequence is written to a `.seq` file using `seq.write()` — this is the standard format consumed by MRzero for simulation.

In [ ]:
kspace = seq.calculate_kspace()

In [ ]:
k_traj_adc = kspace[0]

In [ ]:
k_traj_adc.shape

In [ ]:
plt.figure()
plt.plot(k_traj_adc[0, :], k_traj_adc[1, :], ".")

In [ ]:
ok, error_report = seq.check_timing()

if ok:
    print("Timing check passed successfully")
else:
    print("Timing check failed. Error listing follows:")
    [print(e) for e in error_report]

plt.figure()
seq.plot(time_range=(0.0, 50))

seq.write(seq_filename, v141_compat=True)

## Phantom and Simulation
 
MRzero simulates the MR signal from a digital phantom — a numerical object containing spatially resolved maps of the tissue parameters T1, T2, T2*, and proton density (PD). The phantom used here is a standard brain-like test object that you can inspect with `phantom.plot()` before running the simulation.
 
The simulation proceeds in three steps:
 
1. `phantom.build()` — converts the phantom into the internal tensor format used by MRzero.
2. `mr0.compute_graph()` — traces the sequence operations and k-space trajectory into a differentiable computational graph.
3. `mr0.execute_graph()` — evaluates the graph and returns the complex-valued MR signal for each ADC sample.

The output `signal` is a 1D array of complex numbers containing all acquired k-space samples, ordered by phase-encoding line.

In [ ]:
from typing import Tuple

import torch


def coil_sensitivity(
    grid_size: Tuple[int, int, int],
    coil_orientation: Tuple[float, float, float],
    coil_position: Tuple[float, float, float],
    fov_scaling: Tuple[float, float, float],
) -> torch.Tensor:
    """
    Compute the coil sensitivity map.

    Coordinate positions:
    (-0.5, -0.5) lower left corner
    (+0.5, +0.5) upper right corner

    Parameters
    ----------
    grid_size : Tuple[int, int, int]
        Grid size in each dimension (nx, ny, nz).
    coil_orientation : Tuple[float, float, float]
        Coil orientation (mx, my, mz).
    coil_position : Tuple[float, float, float]
        Coil position (px, py, pz).
    fov_scaling : Tuple[float, float, float]
        Field of view scaling factors.

    Returns
    -------
    torch.Tensor
        Complex-valued coil sensitivity map of shape (nx, ny, nz).
    """
    sqrt3_over_2 = torch.sqrt(torch.tensor(3.0)) / 2

    # Create meshgrid for spatial positions
    rx, ry, rz = torch.meshgrid(
        torch.linspace(
            -sqrt3_over_2 * fov_scaling[0], sqrt3_over_2 * fov_scaling[0], grid_size[0]
        )
        + coil_position[0],
        torch.linspace(
            -sqrt3_over_2 * fov_scaling[1], sqrt3_over_2 * fov_scaling[1], grid_size[1]
        )
        + coil_position[1],
        torch.linspace(
            -sqrt3_over_2 * fov_scaling[2], sqrt3_over_2 * fov_scaling[2], grid_size[2]
        )
        + coil_position[2],
        indexing="ij",
    )

    # Handle degenerate dimensions where grid size is 1
    rx = torch.where(torch.tensor(grid_size[0]) == 1, torch.zeros_like(rx), rx)
    ry = torch.where(torch.tensor(grid_size[1]) == 1, torch.zeros_like(ry), ry)
    rz = torch.where(torch.tensor(grid_size[2]) == 1, torch.zeros_like(rz), rz)

    # Compute radial distance
    r = torch.sqrt(rx**2 + ry**2 + rz**2)

    # Compute magnetic field components
    dot_product = (
        coil_orientation[0] * rx + coil_orientation[1] * ry + coil_orientation[2] * rz
    )
    Bx = 3 * rx * dot_product / r**5 - coil_orientation[0] / r**3
    By = 3 * ry * dot_product / r**5 - coil_orientation[1] / r**3

    # Compute complex-valued sensitivity map
    sensitivity_map = -Bx + 1j * By

    return sensitivity_map


def cyclic_head_coil(
    coil_config: Tuple[int, int, int, int],
    fov_scaling: Tuple[float, float, float] = (1, 1, 1),
    coil_radius: float = 2,
    coil_position_z: float = 0,
    coil_orientation_z: float = 0,
    phase_offset: float = 0,
) -> torch.Tensor:
    """
    Generate the coil sensitivity map for a cyclic head coil configuration.

    Parameters
    ----------
    coil_config : Tuple[int, int, int, int]
        Number of coils and grid size (num_coils, nx, ny, nz).
    fov_scaling : Tuple[float, float, float]
        Field of view scaling factors.
    coil_radius : float
        Radius of cyclic head coil.
    coil_position_z : float
        Coil position along the z-axis.
    coil_orientation_z : float
        Coil orientation along the z-axis.
    phase_offset : float
        Phase offset for coil positioning.

    Returns
    -------
    torch.Tensor
        Complex-valued coil sensitivity map of shape (num_coils, nx, ny, nz).
    """
    num_coils, nx, ny, nz = coil_config

    # Initialize the coil sensitivity map as a complex tensor
    sensitivity_map = torch.zeros(coil_config, dtype=torch.complex128)

    # Generate phase angles for coil positioning
    phase_angles = torch.linspace(0, 2 * torch.pi, num_coils + 1)[:-1] + phase_offset

    # Compute coil sensitivity for each coil
    for coil_idx, phi in enumerate(phase_angles):
        # Define coil orientation
        coil_orientation = (torch.cos(phi), torch.sin(phi), coil_orientation_z)
        # Define coil position
        coil_position = (
            torch.cos(phi) * coil_radius,
            torch.sin(phi) * coil_radius,
            coil_position_z,
        )

        # Compute coil sensitivity map for this coil
        sensitivity_map[coil_idx, :, :, :] = coil_sensitivity(
            (nx, ny, nz),
            coil_orientation,
            coil_position,
            fov_scaling,
        )

    return sensitivity_map


In [ ]:
# !wget https://github.com/MRsources/MRzero-Core/raw/main/documentation/playground_mr0/subject05.npz &> /dev/null

# (i) load a phantom object from file
obj_p = mr0.VoxelGridPhantom.brainweb('subject05.npz')
obj_p = obj_p.interpolate(n_x, n_y, 32).slices([15])
obj_p.size[2]=0.08
phantom = obj_p
phantom.B0 = phantom.B0 * 0.5

In [ ]:
C = cyclic_head_coil((8, n_x, n_y, 1))

In [ ]:
phantom.coil_sens = C

In [ ]:
phantom.coil_sens.shape

In [ ]:
plt.figure()
phantom.plot()

In [ ]:
mr0_seq = mr0.Sequence.import_file(seq_filename)

data = phantom.build()
graph = mr0.compute_graph(
    mr0_seq,
    data,
)
signal = mr0.execute_graph(
    graph,
    mr0_seq,
    data,
    print_progress=True,
)

## Task 5: Reconstruction and Display

The raw signal returned by the simulation contains k-space samples ordered line by line, from the most negative to the most positive phase-encoding step. Reconstruction requires reorganizing this data into a 2D k-space matrix and transforming it back into image space via the inverse Fourier relationship between k-space and image space.
 
**Your tasks:**
 
1. **Convert** the signal from a PyTorch tensor to a NumPy array using `.numpy()`.
2. **Reshape** the 1D signal array into a 2D k-space matrix of shape `(n_y, n_x)`. Each row corresponds to one phase-encoding line.
3. **Reconstruct** the image by applying a 2D inverse Fourier transform. The standard approach in MRI is:
   - `np.fft.ifftshift()` on the k-space matrix — reorders the data so that the DC component (k=0) is in the corner as expected by NumPy's FFT convention.
   - `np.fft.ifft2()` — applies the 2D inverse DFT.
   - `np.fft.fftshift()` on the result — moves the DC component back to the center of the image for display.
   Store the result in a variable called `image`.
4. **Display** the magnitude image using `plt.imshow()`. Take the magnitude with `np.abs()` and use `cmap="gray"`. Set the title to reflect the currently selected contrast weighting using the `case` variable.

Once reconstruction is working, change the `case` variable at the top of the notebook to each of the three contrast modes (T1-weighted, T2-weighted, PD-weighted) and re-run. Observe how the image contrast changes and verify that it matches your expectations from the GRE signal equation.

In [ ]:
# Your solution here

data = signal.numpy().copy()

data = data.reshape((n_y, n_x, 8))

# Revert every second line
data[::2, :, :] = data[::2, ::-1, :]

image = np.fft.fftshift(np.fft.ifftn(np.fft.ifftshift(data, axes=(0, 1)), axes=(0, 1)), axes=(0, 1))

plt.figure()
plt.imshow(np.abs(image[..., 2]), cmap="gray")
plt.title(f"Reconstructed GRE Image — {case}")
plt.show()

In [ ]:
# Your solution here

gy_blip_acc = pp.make_trapezoid(
    channel="y",
    area=2 * delta_ky,
    system=system,
)

seq_acc = pp.Sequence(system)

seq_acc.add_block(rf)

seq_acc.add_block(gx_pre, gy_pre)

# seq.add_block(pp.make_delay(te_delay))

for index in range(0, n_y // 2, 2):

    seq_acc.add_block(gx, adc)

    seq_acc.add_block(gy_blip_acc)

    seq_acc.add_block(gx_rev, adc)

    seq_acc.add_block(gy_blip_acc)

# seq.add_block(gx_spoiler)

# seq.add_block(pp.make_delay(tr_delay))

In [ ]:
seq_acc_filename = "gre_acc.seq"

In [ ]:
seq_acc.write(seq_acc_filename, v141_compat=True)

In [ ]:
mr0_seq_acc = mr0.Sequence.import_file(seq_acc_filename)

data = phantom.build()
graph = mr0.compute_graph(
    mr0_seq_acc,
    data,
)
signal = mr0.execute_graph(
    graph,
    mr0_seq_acc,
    data,
    print_progress=True,
)

In [ ]:
# Your solution here

data = np.zeros((n_y, n_x, 8), dtype=np.complex128)

data[::2, :] = signal.numpy().copy().reshape((n_y//2, n_x, 8))

# Revert every second line
data[::4, :, :] = data[::4, ::-1, :]

image = np.fft.fftshift(np.fft.ifftn(np.fft.ifftshift(data, axes=(0, 1)), axes=(0, 1)), axes=(0, 1))

plt.figure()
plt.imshow(np.abs(image[:, :, 0]), cmap="gray")
plt.title(f"Reconstructed GRE Image — {case}")
plt.show()

In [ ]:
# Parallel imaging + Fourier transform + Sampling Operator + Conjugate Gradient